# CRM Domain Fine-Tuning with LoRA

Fine-tune Mistral 7B with LoRA on CRM domain Q&A data.
Compare fine-tuned model vs prompt-only baseline using RAGAS metrics.

**Requirements:** Run on Google Colab with T4 GPU (free tier works).

## 1. Setup & Install

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl
!pip install -q ragas langchain langchain-openai
!pip install -q huggingface_hub

In [ ]:
import os
from getpass import getpass

# Paste your tokens when prompted — not stored in the notebook
HF_TOKEN = getpass("Hugging Face token: ")
OPENAI_API_KEY = getpass("OpenAI API key (for RAGAS evaluation): ")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from huggingface_hub import login
login(token=HF_TOKEN)

## 2. CRM Training Data

Q&A pairs from the CRM Agentic Reasoning Engine evaluation set.
Covers CRM data queries, documentation, and entity relationships.

In [ ]:
TRAINING_DATA = [
    # --- CRM Data Queries (SQL-grounded) ---
    {
        "instruction": "What deals are in the pipeline?",
        "response": "Current pipeline has 11 open deals totaling approximately $194K. Key deals include Beta Tech new business ($32K, Negotiation), Harbor Logistics ($26K, Discovery), Acme renewal ($24K, Qualified), and Delta Health renewal ($22K, Qualified)."
    },
    {
        "instruction": "How are deals distributed across stages?",
        "response": "Qualified has the most open deals (4): Acme renewal, Delta renewal, Fusion upsell, Crown expansion. Discovery has 3: Beta addon, Delta expand, Harbor new. Proposal has 2: Acme upgrade and Crown renewal. Negotiation has 1: Beta Tech new business. New has 1: Eastern Travel trial."
    },
    {
        "instruction": "Which renewals are at risk?",
        "response": "Two renewals are at risk. Delta Health (renewal Feb 15, at-risk-renewal-soon) has HIPAA audit logging concerns and needs board approval for expansion. Crown Foods (renewal Feb 14, at-risk-renewal-soon) has mobile app performance concerns despite strong product adoption. Combined at-risk renewal value is approximately $40K."
    },
    {
        "instruction": "Tell me more about Crown Foods",
        "response": "Crown Foods is an at-risk SMB on the Pro plan with renewal on Feb 14. They have a subscription renewal opportunity ($18K, Proposal) and a warehouse expansion ($14K, Qualified). Key activities include a renewal check-in with Maria Silva and advanced training. Maria expressed frustration with mobile app performance."
    },
    {
        "instruction": "What are the largest deals in the pipeline?",
        "response": "The largest deals in the pipeline are Beta Tech new rollout ($32K, Negotiation), Harbor Logistics initial rollout ($26K, Discovery), Acme Manufacturing annual renewal ($24K, Qualified), and Delta Health renewal ($22K, Qualified)."
    },
    # --- Act! CRM Documentation (RAG-grounded) ---
    {
        "instruction": "What are the prerequisites for installing Act! Premium with web access?",
        "response": "Act! Premium requires Microsoft Internet Information Services (IIS) 6 and ASP.NET running in 32-bit compatibility mode. You must also sign on as a Microsoft Windows Administrator and it is recommended to disable software-based firewalls during installation."
    },
    {
        "instruction": "How do you register and activate the Act! software after installation?",
        "response": "Go to the Help menu, click Register Act!, select a registration option, click Next, follow the on-screen instructions, and click Activate when finished. On Windows Vista and Windows 7, you must run the software as an Administrator to register it."
    },
    {
        "instruction": "What is Act! Marketing Automation and what are its main features?",
        "response": "Act! Marketing Automation (AMA) is a marketing platform included with Act! Growth Suite. Its main features include Drip Marketing Campaigns for automated scheduled emails, over 170 pre-built email templates with drag-and-drop editor, Landing Pages for lead capture, and tools for GDPR compliance, spam analysis, and managing opt-outs."
    },
    {
        "instruction": "What is a Landing Page in Act! Marketing Automation and how does it work?",
        "response": "A Landing Page is an online web form that contacts can use to register their interest in your business. When a contact completes the form, their details are automatically sent to your Act! database without further interaction from you."
    },
    {
        "instruction": "How do you create a new email template in Act! Marketing Automation?",
        "response": "Go to the Act! Marketing Automation section, click Templates, then click the Actions menu and choose New Template. The template editor uses a Drag and Drop block-based system where you can customize content blocks, add rows, and modify existing elements."
    },
    {
        "instruction": "How do you import contacts into Act! CRM?",
        "response": "Act! provides import functionality to bring contact data into the database. You can import contacts from CSV files and other formats, mapping fields to corresponding Act! database fields during the import process."
    },
    {
        "instruction": "What is a suppression list in Act! Marketing Automation and when do you need one?",
        "response": "A suppression list contains contacts who have unsubscribed from marketing emails. If you have ever used any form of emarketing software, you need to import the details of any contacts who unsubscribed into Act! Marketing Automation. You import suppression lists from CSV files and should enter a source description for tracking."
    },
    {
        "instruction": "How do email alert notifications work in Act! Marketing Automation?",
        "response": "AMA provides several alert types including Accounting Alerts and Email Click Alerts. Email click alerts are generated from the Outlook or Gmail plug-in for one-to-one emails, notifying you instantly when a recipient opens the email. If opened multiple times within 8 hours, there is a suppression period before the next notification."
    },
    {
        "instruction": "What can you do with Act! CRM according to the quick start guide?",
        "response": "Act! helps organize all your prospect and customer details in one place so you can prioritize your day and market your products and services more effectively. Key features include creating contacts, managing lists, tracking opportunities, importing data, and marketing automation."
    },
    {
        "instruction": "How do you add users to an Act! database and what are the limitations?",
        "response": "Go to the Tools menu and click Manage Users. You can assign security roles and permissions to limit access to data and features. You can add any number of users, but you are limited by your license for the number of active users."
    },
    {
        "instruction": "What are Drip Marketing Campaigns in Act! Marketing Automation?",
        "response": "Drip Marketing Campaigns are advanced automated email marketing campaigns that go beyond conventional email blasts. You can send emails on a schedule - the first email when a user subscribes, the second two days later, another after a fortnight, and so on."
    },
    {
        "instruction": "How do you handle duplicate contacts when setting up a Landing Page?",
        "response": "There is an Allow Duplicates setting when creating a Landing Page. Allowing duplicates means the same person could fill in the form multiple times and each time a new contact would be created. It is recommended not to allow duplicates."
    },
    {
        "instruction": "What mandatory elements must be included in marketing email templates?",
        "response": "Marketing email templates must include an Unsubscribe option and company details so contacts know who is contacting them. The template editor includes a default row at the bottom for the unsubscribe link and company information. This is mandatory for marketing emails but can be removed for Landing Page templates."
    },
    {
        "instruction": "How do you configure the unsubscribe page in Act! Marketing Automation?",
        "response": "Navigate to the Unsubscribe Designer in the Admin section. You can configure the Unsubscribe Header (displayed below the unsubscribe button) and footer. A typical unsubscribe page has a header like 'Are you sure you want to unsubscribe?' and the footer can list information the contact will miss."
    },
    {
        "instruction": "How do you configure user permissions for Act! Marketing Automation?",
        "response": "To add permissions to a user, click the permission you wish to add from the left hand column, then use the single arrow icon (>) to add it to the user. You must be an AMA Administrator to access the Admin section where user permissions are managed."
    }
]

print(f"Training samples: {len(TRAINING_DATA)}")

## 3. Format Data for Fine-Tuning

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a CRM assistant for Act! CRM. Answer questions about CRM data, "
    "deal pipelines, contacts, and Act! product documentation. "
    "Ground every answer in real data. Never fabricate information."
)

def format_chat(sample):
    """Format as chat template for instruction tuning."""
    return {
        "text": (
            f"<s>[INST] {SYSTEM_PROMPT}\n\n{sample['instruction']} [/INST] "
            f"{sample['response']}</s>"
        )
    }

dataset = Dataset.from_list(TRAINING_DATA)
dataset = dataset.map(format_chat)
print(dataset)
print(f"\nSample:\n{dataset[0]['text'][:300]}...")

## 4. Load Model with 4-bit Quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_ID}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.1f} GB")

## 5. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Fine-Tune with LoRA

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./crm-lora-output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

In [ ]:
# Save the LoRA adapter
ADAPTER_PATH = "./crm-lora-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")

## 7. Generate Answers: Fine-Tuned vs Baseline

In [ ]:
from peft import PeftModel
from transformers import pipeline

# Reload base model for baseline comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

# Fine-tuned model (base + LoRA adapter)
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)


def generate_answer(model, tokenizer, question, max_new_tokens=256):
    """Generate answer from a model."""
    prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\n{question} [/INST] "
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part (after [/INST])
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response


print("Models ready for comparison.")

In [ ]:
# Evaluation questions — held-out subset + training questions
EVAL_QUESTIONS = [
    {
        "question": "What deals are in the pipeline?",
        "reference": "Current pipeline has 11 open deals totaling approximately $194K. Key deals include Beta Tech new business ($32K, Negotiation), Harbor Logistics ($26K, Discovery), Acme renewal ($24K, Qualified), and Delta Health renewal ($22K, Qualified).",
        "context": "The CRM database contains 11 open opportunities across stages: Qualified (4), Discovery (3), Proposal (2), Negotiation (1), New (1). Total pipeline value is $194K."
    },
    {
        "question": "Which renewals are at risk?",
        "reference": "Two renewals are at risk. Delta Health (renewal Feb 15) has HIPAA audit logging concerns. Crown Foods (renewal Feb 14) has mobile app performance concerns. Combined at-risk renewal value is approximately $40K.",
        "context": "Delta Health: at-risk-renewal-soon, renewal Feb 15, HIPAA audit concerns. Crown Foods: at-risk-renewal-soon, renewal Feb 14, mobile app performance issues. Combined value ~$40K."
    },
    {
        "question": "What is Act! Marketing Automation and what are its main features?",
        "reference": "Act! Marketing Automation (AMA) is a marketing platform included with Act! Growth Suite. Its main features include Drip Marketing Campaigns for automated scheduled emails, over 170 pre-built email templates with drag-and-drop editor, Landing Pages for lead capture, and tools for GDPR compliance.",
        "context": "Act! Marketing Automation documentation: AMA is included with Growth Suite. Features: Drip campaigns, 170+ templates, drag-and-drop editor, landing pages, GDPR tools, spam analysis, opt-out management."
    },
    {
        "question": "What are the prerequisites for installing Act! Premium with web access?",
        "reference": "Act! Premium requires Microsoft IIS 6 and ASP.NET running in 32-bit compatibility mode. You must sign on as a Windows Administrator and disable software-based firewalls during installation.",
        "context": "Act! Install Guide: Prerequisites include IIS 6, ASP.NET in 32-bit mode, Windows Administrator login, disable firewalls during install."
    },
    {
        "question": "How do you handle duplicate contacts when setting up a Landing Page?",
        "reference": "There is an Allow Duplicates setting when creating a Landing Page. Allowing duplicates means the same person could fill the form multiple times creating new contacts each time. It is recommended not to allow duplicates.",
        "context": "Landing Page configuration: Allow Duplicates setting controls whether repeat submissions create new contacts. Recommendation: do not allow duplicates."
    },
]

# Generate answers from both models
baseline_answers = []
finetuned_answers = []

print("Generating baseline answers...")
for q in EVAL_QUESTIONS:
    ans = generate_answer(base_model, tokenizer, q["question"])
    baseline_answers.append(ans)
    print(f"  Q: {q['question'][:50]}... -> {len(ans)} chars")

print("\nGenerating fine-tuned answers...")
for q in EVAL_QUESTIONS:
    ans = generate_answer(ft_model, tokenizer, q["question"])
    finetuned_answers.append(ans)
    print(f"  Q: {q['question'][:50]}... -> {len(ans)} chars")

# Show side-by-side comparison
print("\n" + "="*80)
for i, q in enumerate(EVAL_QUESTIONS):
    print(f"\nQ: {q['question']}")
    print(f"\nBaseline: {baseline_answers[i][:200]}...")
    print(f"\nFine-tuned: {finetuned_answers[i][:200]}...")
    print("-"*80)

## 8. RAGAS Evaluation: Fine-Tuned vs Baseline

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from datasets import Dataset as RagasDataset


def build_ragas_dataset(questions, answers):
    """Build a RAGAS-compatible dataset."""
    return RagasDataset.from_dict({
        "question": [q["question"] for q in questions],
        "answer": answers,
        "reference": [q["reference"] for q in questions],
        "contexts": [[q["context"]] for q in questions],
    })


print("Evaluating baseline model...")
baseline_ds = build_ragas_dataset(EVAL_QUESTIONS, baseline_answers)
baseline_results = evaluate(
    dataset=baseline_ds,
    metrics=[faithfulness, answer_relevancy, answer_correctness],
)

print("\nEvaluating fine-tuned model...")
ft_ds = build_ragas_dataset(EVAL_QUESTIONS, finetuned_answers)
ft_results = evaluate(
    dataset=ft_ds,
    metrics=[faithfulness, answer_relevancy, answer_correctness],
)

print("\nEvaluation complete!")

In [ ]:
# Results comparison
print("=" * 60)
print("RAGAS EVALUATION: Fine-Tuned vs Baseline")
print("=" * 60)
print(f"{'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>12} {'Delta':>10}")
print("-" * 60)

for metric in ["faithfulness", "answer_relevancy", "answer_correctness"]:
    base_val = baseline_results[metric]
    ft_val = ft_results[metric]
    delta = ft_val - base_val
    sign = "+" if delta >= 0 else ""
    print(f"{metric:<25} {base_val:>10.3f} {ft_val:>12.3f} {sign}{delta:>9.3f}")

# Composite score (same weights as your RAG comparison)
def composite(results):
    return (
        0.4 * results["answer_relevancy"]
        + 0.4 * results["faithfulness"]
        + 0.2 * results["answer_correctness"]
    )

base_comp = composite(baseline_results)
ft_comp = composite(ft_results)
delta_comp = ft_comp - base_comp
sign = "+" if delta_comp >= 0 else ""

print("-" * 60)
print(f"{'COMPOSITE SCORE':<25} {base_comp:>10.3f} {ft_comp:>12.3f} {sign}{delta_comp:>9.3f}")
print("=" * 60)
print(f"\nComposite weights: 0.4*relevancy + 0.4*faithfulness + 0.2*correctness")
print(f"Fine-tuning {'improved' if delta_comp > 0 else 'did not improve'} overall quality.")

## 9. Save Results

In [ ]:
import json

results_summary = {
    "model": MODEL_ID,
    "lora_config": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "training": {
        "samples": len(TRAINING_DATA),
        "epochs": 3,
        "learning_rate": 2e-4,
    },
    "baseline_scores": dict(baseline_results),
    "finetuned_scores": dict(ft_results),
    "baseline_composite": base_comp,
    "finetuned_composite": ft_comp,
    "improvement": delta_comp,
}

with open("finetune_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to finetune_results.json")
print("\nDownload this file for your records.")

## 10. Optional: Push Adapter to Hugging Face

Upload your fine-tuned LoRA adapter to your Hugging Face account.

In [ ]:
# Uncomment to push to your Hugging Face account
# REPO_NAME = "your-username/crm-mistral-lora"  # Change this
# model.push_to_hub(REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)
# print(f"Adapter pushed to https://huggingface.co/{REPO_NAME}")